# LS-NEQ 05_scale_and_checkpoints — Skalierung und Checkpoint-Sweep

**Dossier-Stufe F** (`lsne.experiments.STAGES["F"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Führt eine Shard aus `jobs/shards.yaml` aus: entweder die Stufen A–E eines Modells/Checkpoints (Shards 1, 2, 3, 5) oder den Stufe-F-Sweep über (Modell, Revision)-Paare (Shard 4: 27 Dossier-Checkpoints 4k…143k von pythia-160m mit B+D; Shard 6: pythia-1b und pythia-1.4b mit B/C/D, nur C1). Jedes Paar wird mit frisch geladenem Modell gerechnet, die Ergebnisse landen pro Stufe sofort auf Drive (inkrementeller Export); ein Wiederaufnahme-Schutz überspringt Paare/Stufen, deren `manifest.json` bereits `status: ok` meldet.

**Eingaben.** `jobs/shards.yaml` (SHARD = 1…6), Stimulusbank; optional eigene Paare über `FORCE_PAIRS`.

**Ausgaben.** Laufverzeichnisse `<RUNS_ROOT>/<stufe>/<modell>@<rev>/` (Shards 1–3, 5) bzw. `<RUNS_ROOT>/F/<modell>@<rev>/shard<i>of<n>/<stufe>/<modell>@<rev>/` (Shards 4, 6) mit `plan`/`runs`-Tabellen; zusammengeführte CSVs unter `.../merged/`; Verlaufsplots über Checkpoints.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/F/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/05_scale_and_checkpoints/`; am Ende eine kurze `summary_05_scale_and_checkpoints_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Shard wählen → Plan (Paare × Familien, Shard-Zuordnung) → Schleife mit Wiederaufnahme-Schutz → Merge (`lsne.cli merge`) → Verlauf über Checkpoints → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Shard wählen (jobs/shards.yaml) ---------------------------------------------------
NOTEBOOK = "05_scale_and_checkpoints"
STAGE = "F"
SHARD = 4                 # 1..6 aus jobs/shards.yaml (4 = Checkpoint-Trace, 6 = 1b/1.4b); 0 = eigene Angaben
SMOKE = False             # True: pythia-14m, 1 Familie, Stufen B+D limitiert (< 3 min CPU)
REUSE = True              # Paare/Stufen mit manifest.json (status ok) überspringen
DTYPE = "float32"
DEVICE = "auto"
FAMILIES = None           # z. B. ["F01", "F02"]; None = alle
SEED = 0
FORCE_PAIRS = None        # z. B. "step4000,step5000" | "EleutherAI/pythia-410m@step143000" | "trace" | None = laut Shard
FORCE_F_STAGES = None     # z. B. ["B", "D"]; None = laut Shard
SHARD_INDEX, SHARD_COUNT = None, None   # Feinaufteilung einer Shard auf mehrere GPUs (None = laut Shard)

import subprocess
import sys

try:
    import yaml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml

with open("jobs/shards.yaml", encoding="utf-8") as f:
    PLAN = yaml.safe_load(f)
shards = {int(s["id"]): s for s in PLAN["shards"]}
shard = shards.get(int(SHARD), dict(id=0, account="-", gpu="-", purpose="manuell (FORCE_PAIRS / FORCE_F_STAGES)",
                                    model="EleutherAI/pythia-160m", revision="step98000", stages=["A", "B", "C", "D", "E"]))
print(f"Shard {shard['id']} ({shard.get('account', '?')} / {shard.get('gpu', '?')}): {shard.get('purpose', '')}")
print("Stufen:", shard.get("stages"), "| F-Stufen:", shard.get("f_stages"), "| C-Stufen:", shard.get("c_levels"), "| SAE:", shard.get("sae"))
print("CLI-Befehl der Shard:\n  " + " ".join(str(shard.get("command", "-")).split()))
print("Alle Shards:")
for s in PLAN["shards"]:
    print(f"  {s['id']}: {s.get('account')}/{s.get('gpu')} — {s.get('purpose')}")


In [ ]:
# --- Plan: Paare (Modell, Revision) × Familien mit deterministischer Shard-Zuordnung ------
from dataclasses import replace

import numpy as np
import pandas as pd

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C
from lsne.experiments.stage_f_scale import family_filter, parse_pairs, plan_triples

base_model = shard.get("model") or (shard.get("models") or ["EleutherAI/pythia-160m"])[0]
base_rev = str(shard.get("revision", "step143000"))
spec = ModelSpec(name=base_model, revision=base_rev, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
if shard.get("c_levels"):
    cfg.c_levels = tuple(str(c) for c in shard["c_levels"])
limits = C.Limits()
limits.use_sae = bool(shard.get("sae"))
IS_F = "F" in list(shard.get("stages") or [])
if IS_F:
    if shard.get("revisions") == "config.CHECKPOINT_TRACE":
        pairs = parse_pairs("trace", cfg)
    elif shard.get("models"):
        pairs = [(str(m), base_rev) for m in shard["models"]]
    else:
        pairs = parse_pairs(None, cfg)
    f_stages = [str(s) for s in (shard.get("f_stages") or limits.f_stages)]
    shard_index, shard_count = int(shard.get("shard_index", 0)), int(shard.get("shard_count", 1))
else:
    pairs = [(base_model, base_rev)]
    f_stages = [str(s) for s in (shard.get("stages") or ["A", "B", "C", "D", "E"])]
    shard_index, shard_count = 0, 1
if FORCE_PAIRS:
    pairs = parse_pairs(FORCE_PAIRS, cfg)
    IS_F = True
if FORCE_F_STAGES:
    f_stages = [str(s).upper() for s in FORCE_F_STAGES]
if SHARD_INDEX is not None and SHARD_COUNT is not None:
    shard_index, shard_count = int(SHARD_INDEX), int(SHARD_COUNT)
if SMOKE:
    spec = ModelSpec(name="EleutherAI/pythia-14m", revision="step98000", dtype="float32", device=DEVICE)
    cfg = C.smoke_config(replace(cfg, model=spec, families=FAMILIES or ["DEV01"]))
    limits = C.Limits.smoke()
    pairs, f_stages, IS_F = [(spec.name, spec.revision)], ["B", "D"], True
# the per-pair stages of the sweep run with the reduced ("limited") settings like CLI stage F
sub_limits = limits.as_limited() if IS_F else limits
families = sorted({f.family_id for f in C.select_families(cfg)})
plan = plan_triples(pairs, families, shard_index, shard_count)
mine = plan[plan["mine"]]
CLI_COMMAND = " ".join(str(shard.get("command", "")).split()) or "python -m lsne.cli run ..."
print(f"{len(pairs)} Paare × {len(families)} Familien = {len(plan)} Tripel, davon {len(mine)} in Shard {shard_index}/{shard_count}; "
      f"Stufen pro Paar: {f_stages}; Layout: {'F-Sweep' if IS_F else 'Standard'}; SMOKE={SMOKE}")
print("Paare:", pairs[:8], "..." if len(pairs) > 8 else "")


In [ ]:
# --- Schleife über die Paare: frisches Modell je Paar, inkrementeller Drive-Export, Resume --
import contextlib
import datetime
import gc
import json
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, read_manifest, save_df, write_manifest
from lsne.model import load_model

FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def show(df, n: int = 40):
    df = df.head(n) if isinstance(df, pd.DataFrame) else df
    try:
        display(df)          # noqa: F821
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_shard{shard['id']}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


def manifest_ok(rd) -> bool:
    try:
        return rd.manifest_path.exists() and read_manifest(rd).get("status") == "ok"
    except Exception:
        return False


if IS_F:
    f_dir = RunDir(cfg.out_root, "F", cfg.model)                      # same layout as `lsne.cli run --stage F`
    sub_root = f_dir.path / f"shard{shard_index}of{shard_count}"
else:
    f_dir, sub_root = None, Path(cfg.out_root)
progress_path = os.path.join(OUT_ROOT, f"progress_{NOTEBOOK}_shard{shard['id']}.json")
rows = []
t_all = time.perf_counter()


def flush_progress():
    # the stage tables are already on Drive (finish_stage writes them per stage); this
    # small file records what finished so that a new session can show the state at once
    with open(progress_path, "w", encoding="utf-8") as f:
        json.dump(dict(shard=shard["id"], pairs=pairs, stages=f_stages, rows=rows,
                       timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat()), f, indent=1)


done_pairs = list(dict.fromkeys(zip(mine["model"], mine["revision"])))
for n_pair, (model, revision) in enumerate(done_pairs, 1):
    fams = sorted(mine[(mine["model"] == model) & (mine["revision"] == revision)]["family"].tolist())
    pspec = replace(cfg.model, name=model, revision=revision)
    sub_cfg = replace(cfg, model=pspec, families=family_filter(cfg.families, fams))
    todo = [st for st in f_stages if not (REUSE and manifest_ok(RunDir(sub_root, st, pspec, mkdir=False)))]
    print(f"\n[{n_pair}/{len(done_pairs)}] {model}@{revision}: Familien {len(fams)}, Stufen {todo or 'alle fertig (Resume)'}", flush=True)
    for st in f_stages:
        if st not in todo:
            rows.append(dict(model=model, revision=revision, stage=st, status="skipped_done", seconds=0.0, n_tables=0,
                             families=",".join(fams), error="", run_dir=str(RunDir(sub_root, st, pspec, mkdir=False).path)))
    if not todo:
        continue
    t_load = time.perf_counter()
    try:
        mb2 = load_model(pspec)
        rows.append(dict(model=model, revision=revision, stage="load", status="ok", seconds=time.perf_counter() - t_load,
                         n_tables=0, families=",".join(fams), error="", run_dir=""))
    except Exception as e:                      # e.g. a checkpoint that does not exist
        traceback.print_exc()
        rows.append(dict(model=model, revision=revision, stage="load", status="error", seconds=time.perf_counter() - t_load,
                         n_tables=0, families=",".join(fams), error=repr(e), run_dir=""))
        flush_progress()
        continue
    ctx: dict = {}
    for st in todo:
        rd = RunDir(sub_root, st, pspec)
        t_st = time.perf_counter()
        print(f"  Stufe {st} ({STAGE_TITLES[st]}) -> {rd.path}", flush=True)
        try:
            tables = STAGES[st].run(sub_cfg, mb2, rd, ctx=ctx, limits=sub_limits)
            rows.append(dict(model=model, revision=revision, stage=st, status="ok", seconds=time.perf_counter() - t_st,
                             n_tables=len(tables), families=",".join(fams), error="", run_dir=str(rd.path)))
            print(f"  Stufe {st} ok in {time.perf_counter() - t_st:.0f} s", flush=True)
        except Exception as e:
            traceback.print_exc()
            C.write_error(rd, st, e, sub_cfg, mb2)
            rows.append(dict(model=model, revision=revision, stage=st, status="error", seconds=time.perf_counter() - t_st,
                             n_tables=0, families=",".join(fams), error=repr(e), run_dir=str(rd.path)))
        flush_progress()
    del mb2, ctx
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

runs = pd.DataFrame(rows, columns=["model", "revision", "stage", "status", "seconds", "n_tables", "families", "error", "run_dir"])
if f_dir is not None:
    save_df(plan, f_dir, "plan")
    save_df(runs, f_dir, "runs")
    write_manifest(f_dir, stage="F", status="ok", cfg=cfg, model=cfg.model, pairs=pairs, stages=f_stages,
                   shard_index=shard_index, shard_count=shard_count, n_triples=int(len(plan)), n_mine=int(len(mine)),
                   sub_root=str(sub_root), notebook=NOTEBOOK, shard_id=shard["id"],
                   runtime_s=time.perf_counter() - t_all)
flush_progress()
print(f"\nFertig in {time.perf_counter() - t_all:.0f} s; Fortschritt: {progress_path}")
show(runs, n=200)


In [ ]:
# --- Zusammenführen (lsne.cli merge) und Verlauf über Checkpoints / Modelle --------------
with guarded("Merge und Verlauf"):
    from lsne.cli import merge_runs

    merged_dir = Path(sub_root) / "merged"
    counts = merge_runs(sub_root, merged_dir)
    print({k: v for k, v in sorted(counts.items())})


    def step_of(rev):
        s = str(rev)
        return int(s[4:]) if s.startswith("step") and s[4:].isdigit() else np.nan


    def load_merged(name):
        p = merged_dir / f"{name}.csv"
        if not p.exists():
            return pd.DataFrame()
        df = pd.read_csv(p)
        df["step"] = df["run_revision"].map(step_of)
        return df


    n_rev = len(set(r for _, r in pairs))
    if IS_F and n_rev > 1:
        # checkpoint trace: lambda (depth injection, lang_dom, largest dose) and baseline log p(target)
        dp = load_merged("damage_phase")
        base = load_merged("baseline")
        hy = load_merged("hysteresis")
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        if len(dp):
            sub = dp[(dp["inject"] == "depth") & (dp["dir_family"] == "lang_dom") & (dp["kind"] == "standard")]
            sub = sub[sub["eps"] == sub["eps"].max()]
            for K, g in sorted(sub.groupby("K")):
                m = g.groupby("step")["lam_mean"].mean().sort_index()
                axes[0].plot(m.index, m.values, "o-", ms=3, label=f"K={K}")
            axes[0].set_xscale("log")
            axes[0].set_xlabel("Trainingsschritt")
            axes[0].set_ylabel("λ (lang_dom, Injektion depth, ε max)")
            axes[0].set_title("Störungswachstum über das Training")
            axes[0].legend(fontsize=7)
        if len(base):
            bs = base[base["kind"] == "standard"] if "kind" in base else base
            for K, g in sorted(bs.groupby("K")):
                m = g.groupby("step")["logp_target"].mean().sort_index()
                axes[1].plot(m.index, m.values, "o-", ms=3, label=f"K={K}")
            axes[1].set_xscale("log")
            axes[1].set_xlabel("Trainingsschritt")
            axes[1].set_ylabel("ln p₀(Ziel)")
            axes[1].set_title("Baseline über das Training")
            axes[1].legend(fontsize=7)
        if len(hy):
            for (K, d), g in sorted(hy.groupby(["K", "direction"])):
                m = g.groupby("step")["area_minus_null"].mean().sort_index()
                axes[2].plot(m.index, m.values, "o-", ms=3, label=f"K={K} {d}")
            axes[2].set_xscale("log")
            axes[2].axhline(0, color="k", lw=0.6)
            axes[2].set_xlabel("Trainingsschritt")
            axes[2].set_ylabel("A_hyst − Null")
            axes[2].set_title("Hysterese über das Training")
            axes[2].legend(fontsize=6)
        savefig("checkpoint_trace")
        plt.show()
    elif IS_F:
        # scale arm: compare models side by side on the phase table
        dp = load_merged("damage_phase")
        if len(dp):
            sub = dp[(dp["inject"] == "depth") & (dp["dir_family"] == "lang_dom") & (dp["kind"] == "standard")]
            show(sub.pivot_table(index=["run_model", "K"], columns="eps", values="lam_mean"), n=60)
    else:
        print("Stufen A-E eines Modells: Abbildungen in den Notebooks 00-04 (REUSE=True lädt diese Ergebnisse).")


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, shard=shard["id"], purpose=shard.get("purpose", ""), smoke=bool(SMOKE),
        model=spec.name, revision=spec.revision, dtype=spec.dtype, pairs=pairs, stages=f_stages, layout=("F" if IS_F else "standard"),
        shard_index=shard_index, shard_count=shard_count, families=families, n_triples=int(len(plan)), n_mine=int(len(mine)),
        out_root=OUT_ROOT, runs_root=RUNS_ROOT, sub_root=str(sub_root), cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        runs=runs.to_dict("records"),
        n_ok=int((runs["status"] == "ok").sum()), n_error=int((runs["status"] == "error").sum()),
        n_skipped=int((runs["status"] == "skipped_done").sum()),
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_shard{shard['id']}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps({k: v for k, v in C.to_jsonable(summary).items() if k != "runs"}, indent=1, ensure_ascii=False)[:3000])
